In [2]:
import pandas as pd

# 1. Load the dataset you just uploaded
df = pd.read_csv('oasis_cross-sectional.csv')

# 2. Drop columns that are completely irrelevant to the model prediction
df = df.drop(['ID', 'Delay'], axis=1, errors='ignore')

# 3. Fill missing 'Educ' and 'SES' values with the median of those columns
df['Educ'] = df['Educ'].fillna(df['Educ'].median())
df['SES'] = df['SES'].fillna(df['SES'].median())

# 4. Drop any remaining rows that might have missing target variables (CDR)
df = df.dropna(subset=['CDR'])

# 5. Binarize the target variable (CDR) into an Alzheimer's Risk Score
# If CDR is 0, Risk is 0 (No Alzheimer's). If CDR is greater than 0, Risk is 1 (Alzheimer's Risk)
df['Alzheimers_Risk'] = (df['CDR'] > 0).astype(int)

# 6. Check our work to make sure we have a perfectly clean dataset
print("Missing values after cleaning:")
print(df.isnull().sum())

Missing values after cleaning:
M/F                0
Hand               0
Age                0
Educ               0
SES                0
MMSE               0
CDR                0
eTIV               0
nWBV               0
ASF                0
Alzheimers_Risk    0
dtype: int64


In [3]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import joblib

# 1. Select our specific features (X) and our target (y)
X = df[['Age', 'Educ', 'SES', 'MMSE', 'eTIV', 'nWBV', 'ASF']]
y = df['Alzheimers_Risk']

# 2. Split the data (80% for training, 20% for testing)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Initialize and train the Random Forest Model
model = RandomForestClassifier(max_depth=5, random_state=42)
model.fit(X_train, y_train)

# 4. Test how accurate our new model is
predictions = model.predict(X_test)
accuracy = accuracy_score(y_test, predictions)
print("Model Successfully Trained!")
print(f"Accuracy on unseen test data: {accuracy * 100:.2f}%")

# 5. Save the trained model to a file so the dashboard can use it
joblib.dump(model, 'model.pkl')
print("Model saved as 'model.pkl' in your Colab files!")

Model Successfully Trained!
Accuracy on unseen test data: 89.36%
Model saved as 'model.pkl' in your Colab files!


In [4]:
!pip install gradio

In [5]:
import gradio as gr
import pandas as pd
import joblib

# 1. Load the Alzheimer's prediction model you just trained
model = joblib.load('model.pkl')

# 2. Define the Backend Engine (calculates the Alzheimer's Risk Score)
def predict_alzheimers_risk(age, educ, ses, mmse, etiv, nwbv, asf):
    # Package inputs into a DataFrame that matches your training data
    input_data = pd.DataFrame([[age, educ, ses, mmse, etiv, nwbv, asf]],
                              columns=['Age', 'Educ', 'SES', 'MMSE', 'eTIV', 'nWBV', 'ASF'])

    # Get the probability of Alzheimer's (Class 1)
    risk_prob = model.predict_proba(input_data)[0][1]
    risk_percentage = round(risk_prob * 100, 2)

    # Map the percentage to a clinical risk category
    if risk_percentage < 25:
        category = "🟢 Low Risk"
    elif risk_percentage < 60:
        category = "🟡 Moderate Risk"
    else:
        category = "🔴 High Risk"

    return f"{risk_percentage}% - {category}"

# 3. Build the Frontend Web Interface
interface = gr.Interface(
    fn=predict_alzheimers_risk,
    title="Multimodal Alzheimer’s Disease Risk Assessment Dashboard",
    description="Enter patient demographics, clinical scores, and brain imaging metrics below to generate a real-time Alzheimer's risk assessment.",
    inputs=[
        gr.Slider(minimum=40, maximum=100, step=1, value=75, label="Age (Years)"),
        gr.Slider(minimum=6, maximum=20, step=1, value=12, label="Years of Education (Educ)"),
        gr.Dropdown(choices=[1, 2, 3, 4, 5], value=3, label="Socioeconomic Status (SES: 1=Highest, 5=Lowest)"),
        gr.Slider(minimum=0, maximum=30, step=1, value=25, label="Mini-Mental State Exam Score (MMSE)"),
        gr.Number(value=1500, label="Estimated Total Intracranial Volume (eTIV)"),
        gr.Slider(minimum=0.6, maximum=0.9, step=0.01, value=0.75, label="Normalized Whole Brain Volume (nWBV)"),
        gr.Number(value=1.2, label="Atlas Scaling Factor (ASF)")
    ],
    outputs=gr.Textbox(label="Alzheimer's Risk Score"),
    theme="default"
)

# 4. Launch the application!
interface.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://448e126277155bf679.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
